# 01 — Quickstart: Load a model and generate text

The two-step workflow for every inference task:

```
LLM.from_pretrained()  →  LLM.generate_text()
```

`from_pretrained` handles config, weight download, and tokenizer in one call.

> **Colab users:** Run the setup cell, restart the runtime, then continue.


In [1]:
import subprocess, sys, os

def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _is_colab():
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install',
         '--force-reinstall', '--no-cache-dir',
         'git+https://github.com/silvaxxx1/MyLLM.git'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('Install FAILED:', r.stderr[-2000:])
    else:
        print('Done! Now go to Runtime → Restart runtime, then continue.')
else:
    root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    has_uv = subprocess.run(['which', 'uv'], capture_output=True).returncode == 0
    cmd = ['uv', 'pip', 'install', '-e', root] if has_uv else [sys.executable, '-m', 'pip', 'install', '-e', root]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('Install FAILED:', r.stderr[-2000:])
    else:
        print('Local editable install ready.')


Local editable install ready.


## 1. Inspect a ModelConfig

In [2]:
import torch
from myllm import ModelConfig

cfg = ModelConfig.from_name('gpt2-small')

print(f'Model      : {cfg.name}')
print(f'Layers     : {cfg.n_layer}')
print(f'Heads      : {cfg.n_head}')
print(f'Embedding  : {cfg.n_embd}')
print(f'Vocab size : {cfg.vocab_size}')
print(f'Block size : {cfg.block_size}')

mem = cfg.estimate_memory(batch_size=1, dtype=torch.float32)
print(f'\nMemory estimate (fp32):')
print(f'  Parameters : {mem["n_parameters"]/1e6:.1f}M  ({mem["parameters_gb"]:.2f} GB)')
print(f'  Activations: {mem["activations_gb"]:.3f} GB')
print(f'  Total      : {mem["total_gb"]:.2f} GB')

2026-05-09 20:08:42.572001: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Model      : gpt2-small
Layers     : 12
Heads      : 12
Embedding  : 768
Vocab size : 50257
Block size : 1024

Memory estimate (fp32):
  Parameters : 151.9M  (0.57 GB)
  Activations: 0.141 GB
  Total      : 0.71 GB


## 2. Load model + tokenizer in one line

`LLM.from_pretrained()` downloads weights on first run, caches them locally,
maps them to the GPT architecture, and auto-loads the matching tokenizer.

> **On Colab:** weights are cached in `/content/models/` for the session.


In [3]:
from myllm import LLM

llm = LLM.from_pretrained('gpt2-small')
print(llm)  # LLM(model='gpt2-small', params=..., device='...', dtype=...)


📋 Using custom config
📊 Estimated memory: 0.71 GB
   Parameters: 0.57 GB
   Activations: 0.14 GB
⬇️  Downloading gpt2-small...
The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.
✅ File ./models/model-gpt2-small.safetensors already exists and is valid.
📂 Loading weights from disk...
📦 Loading weights from ./models/model-gpt2-small.safetensors to cpu...
✅ Loaded 160 tensors
🏗️  Initializing model architecture...
🔧 Converting model to torch.float32...
🔄 Mapping weights using gpt2_mapper...
Loading gpt2-small weights /

Loading GPT-2 blocks:   0%|          | 0/12 [00:00<?, ?block/s]

Loading gpt2-small weights -

Loading GPT-2 blocks:   8%|▊         | 1/12 [00:01<00:17,  1.63s/block]

Loading gpt2-small weights /

...

Loading GPT-2 blocks:  83%|████████▎ | 10/12 [00:07<00:00,  2.53block/s]

Loading gpt2-small weights /

Loading GPT-2 blocks: 100%|██████████| 12/12 [00:07<00:00,  1.58block/s]

Loading gpt2-small weights \

✅ Successfully loaded gpt2-small!
🎯 Model ready on cuda for inference!
LLM(model='gpt2-small', params=163.0M, device='cuda', dtype=torch.float32)


## 3. `generate_text()` — string in, string out

The tokenizer is already bundled in `llm.tokenizer`. No need to pass it explicitly.
Use `skip_prompt=True` to get only the newly generated text (without the input prompt).


In [4]:
from myllm import GenerationConfig

gen_cfg = GenerationConfig(
    max_length=60,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    do_sample=True,
    use_kv_cache=True,
)

# tokenizer arg omitted — uses llm.tokenizer automatically
result = llm.generate_text('The future of AI is', generation_config=gen_cfg)
print('Full output (prompt + generated):')
print(result['text'])

print()

# skip_prompt=True returns only the new tokens
result = llm.generate_text('The future of AI is', generation_config=gen_cfg, skip_prompt=True)
print('Generated only (no prompt):')
print(result['text'])


Full output (prompt + generated):
The future of AI is bright, but what will happen to its future if we continue to have AI?

How will AI become a "new age"

Why do AI scientists still care so much about the past?

How do AI scientists see the future of medicine?

How is AI the best

Generated only (no prompt):
 bright, but for now we are just getting started. If you are curious about the future of AI, this article will give you some information on the past and present of AI.

Why is AI Important?

I've heard about AI and I'm going to try to explain why.


## 4. `generate()` — token tensor in, token tensor out

For lower-level control: encode manually, call `generate()`, decode with `llm.tokenizer`.


In [5]:
import torch
from myllm import GenerationConfig

device = llm.device
prompt = 'In a world where machines think,'
input_ids = llm.tokenizer.encode(prompt, return_tensors='pt').to(device)
print('Prompt shape:', input_ids.shape)  # (1, seq_len)

gen_cfg = GenerationConfig(
    max_length=40,
    temperature=1.0,
    top_k=50,
    do_sample=True,
    use_kv_cache=True,
    return_logprobs=True,
)

output = llm.generate(input_ids, gen_cfg)

print('Output tokens shape :', output['tokens'].shape)
print('Log-probs shape     :', output['logprobs'].shape)
print()
print(llm.tokenizer.decode(output['tokens'][0], skip_special_tokens=True))


Prompt shape: torch.Size([1, 7])
Output tokens shape : torch.Size([1, 47])
Log-probs shape     : torch.Size([1, 40])

In a world where machines think, the world is not so tidy.

A new research by Princeton University researchers suggests that machine learning could be a kind of natural language. It's the theory of logical reasoning that allows the learning of


## 5. `generate_batch()` — multiple prompts at once


In [6]:
from myllm import GenerationConfig

prompts = [
    'Once upon a time',
    'The capital of France is',
    'To train a neural network you need',
]

gen_cfg = GenerationConfig(
    max_length=30,
    temperature=0.9,
    top_k=40,
    do_sample=True,
    use_kv_cache=False,  # KV cache not supported with padded batches
)

results = llm.generate_batch(prompts, llm.tokenizer, gen_cfg)

for i, res in enumerate(results):
    print(f'[{i}] {res["text"]}')
    print()


[0] Once upon a time!!! It's all happening so fast! Now I hope we're still in the fight! " -Odin, speaking to the hero after seeing him die

[1] The capital of France is!!

A word of warning though. The city that we were here to stay for is the most beautiful. It contains the oldest art in Europe,

[2] To train a neural network you need to know if there's anything else you can do to improve the accuracy of your input. If you find something, I suggest taking a look at the



In [7]:
print('Models cached on disk:')
for family, variants in llm.list_models().items():
    for v in variants:
        print(f'  {v}')


Models cached on disk:
  gpt2-small
  gpt2-medium
  gpt2-large
  gpt2-xl
  llama2-7b
  llama2-13b
  llama3-1b
  llama3-8b
  mistral-7b-v0.1
  phi-2
  gemma-2b
  gemma-7b
